In [77]:
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from scipy.interpolate import interp1d
from scipy.stats import mode

In [2]:
import pandas as pd
import numpy as np
import os

def load_wesad_data(directory):
    data = {}
    for participant_id in os.listdir(directory):
        participant_path = os.path.join(directory, participant_id)
        if os.path.isdir(participant_path):
            participant_data = {}
            for file_name in os.listdir(participant_path):
                file_path = os.path.join(participant_path, file_name)
                if file_name.endswith('.pkl'):
                    participant_data[file_name.split('.')[0]] = pd.read_pickle(file_path)
            data[participant_id] = participant_data
    return data

wesad_data = load_wesad_data('C:/Users/rusha/Desktop/Uni_Freiburg_Notes/MDD/WESAD')

In [79]:
wesad_data

{'S10': {'S10': {'signal': {'chest': {'ACC': array([[ 1.12779999,  0.15199995,  0.34159994],
            [ 1.09319997,  0.18879998,  0.29219997],
            [ 1.03539991,  0.20940006,  0.18579996],
            ...,
            [ 0.89419997,  0.03380001, -0.21460003],
            [ 0.89499998,  0.03419995, -0.21820003],
            [ 0.89639997,  0.03260005, -0.22140002]]),
     'ECG': array([[-1.33369446],
            [-1.32774353],
            [-1.32206726],
            ...,
            [ 0.53050232],
            [ 0.53375244],
            [ 0.54057312]]),
     'EMG': array([[-0.01368713],
            [-0.02192688],
            [-0.00901794],
            ...,
            [ 0.00654602],
            [-0.00141907],
            [-0.00814819]]),
     'EDA': array([[0.71601868],
            [0.7144928 ],
            [0.71563721],
            ...,
            [1.70440674],
            [1.74827576],
            [1.72462463]]),
     'Temp': array([[33.69586 ],
            [33.741333],
       

In [88]:
wesad_data['S11']['S11']['label'].value_counts

AttributeError: 'numpy.ndarray' object has no attribute 'value_counts'

In [28]:
wesad_data['S11']['S11']['label'].re

array([0, 0, 0, ..., 0, 0, 0])

In [43]:
(wesad_data['S2']['S2']['label']).shape

(4255300,)

In [14]:
import numpy as np
from scipy.interpolate import interp1d
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

def interpolate_signal(signal, original_rate, target_rate, target_length):
    """
    Interpolates the signal to match the target length.
    
    Parameters:
    - signal: The original signal array.
    - original_rate: Sampling rate of the original signal.
    - target_rate: Desired sampling rate (corresponds to label frequency).
    - target_length: Length of the interpolated signal (number of target samples).
    
    Returns:
    - Interpolated signal.
    """
    original_length = len(signal)
    
    # generate time vectors
    time_original = np.linspace(0, original_length / original_rate, original_length)
    time_target = np.linspace(0, original_length / original_rate, target_length)
    
    # create an interpolation function
    interpolator = interp1d(time_original, signal, kind='linear', fill_value='extrapolate')
    
    # interpolate to the new length
    interpolated_signal = interpolator(time_target)
    
    return interpolated_signal

def extract_and_process_data_wrist(subject_id):
    subject_data = wesad_data[subject_id]
    subject_details = subject_data[subject_id]
    wrist_data = subject_details['signal']['wrist']
    labels = subject_details['label']
    
    # include only valid ones
    valid_labels = np.isin(labels, [0, 1, 2, 3, 4])
    labels = labels[valid_labels]
    
    num_samples = len(labels)  # Number of samples in the labels
    
    # Interpolate each data component to match the length of labels
    bvp_interpolated = interpolate_signal(wrist_data['BVP'].flatten(), sampling_rates['BVP'], sampling_rates['label'], num_samples)
    eda_interpolated = interpolate_signal(wrist_data['EDA'].flatten(), sampling_rates['EDA'], sampling_rates['label'], num_samples)
    temp_interpolated = interpolate_signal(wrist_data['TEMP'].flatten(), sampling_rates['TEMP'], sampling_rates['label'], num_samples)
    
    # reshape each feature to ensure proper concatenation
    bvp_interpolated = bvp_interpolated.reshape(-1, 1)
    eda_interpolated = eda_interpolated.reshape(-1, 1)
    temp_interpolated = temp_interpolated.reshape(-1, 1)
    
    print(f"BVP shape: {bvp_interpola3ted.shape}")
    print(f"EDA shape: {eda_interpolated.shape}")
    print(f"TEMP shape: {temp_interpolated.shape}")
    
    # combine the interpolated features into a single array
    features = np.hstack((
        bvp_interpolated,
        eda_interpolated,
        temp_interpolated
    ))

    return features, labels

# sampling rates
sampling_rates = {
    'BVP': 64,
    'EDA': 4,
    'TEMP': 4,
    'label': 700
}

# extract and process data from subjects
features_s2, labels_s2 = extract_and_process_data_wrist('S2')
features_s3, labels_s3 = extract_and_process_data_wrist('S3')
features_s4, labels_s4 = extract_and_process_data_wrist('S4')
features_s5, labels_s5 = extract_and_process_data_wrist('S5')
features_s6, labels_s6 = extract_and_process_data_wrist('S6')
features_s7, labels_s7 = extract_and_process_data_wrist('S7')
features_s8, labels_s8 = extract_and_process_data_wrist('S8')
features_s9, labels_s9 = extract_and_process_data_wrist('S9')
features_s10, labels_s10 = extract_and_process_data_wrist('S10')
features_s11, labels_s11 = extract_and_process_data_wrist('S11')
features_s13, labels_s13 = extract_and_process_data_wrist('S13')
features_s14, labels_s14 = extract_and_process_data_wrist('S14')
features_s15, labels_s15 = extract_and_process_data_wrist('S15')
features_s16, labels_s16 = extract_and_process_data_wrist('S16')
features_s17, labels_s17 = extract_and_process_data_wrist('S17')

# combine features and labels from both subjects
combined_features = np.vstack((features_s2, features_s3, features_s4, features_s5, features_s6, features_s7, features_s8,features_s9, features_s10, features_s11, features_s13, features_s14, features_s15, features_s16, features_s17))
combined_labels = np.hstack((labels_s2, labels_s3, labels_s4, labels_s5, labels_s6, labels_s7, labels_s8, labels_s9, labels_s10, labels_s11, labels_s13, labels_s14, labels_s15, labels_s16 ,labels_s17))

# combined_features = np.vstack((features_s2))
# combined_labels = np.hstack((labels_s2))

# check the initial label distribution
initial_label_distribution = Counter(combined_labels)
print("Initial label distribution:", initial_label_distribution)

# remove labels 5, 6, 7
valid_labels = np.isin(combined_labels, [0, 1, 2, 3, 4])
balanced_features = combined_features[valid_labels]
balanced_labels = combined_labels[valid_labels]

# determine the minimum number of samples for any class
min_samples_per_class = min(Counter(balanced_labels).values())

# create balanced dataset by randomly sampling min_samples_per_class instances from each class
balanced_features_list = []
balanced_labels_list = []

for label in np.unique(balanced_labels):
    label_indices = np.where(balanced_labels == label)[0]
    if len(label_indices) > min_samples_per_class:
        sampled_indices = np.random.choice(label_indices, min_samples_per_class, replace=False)
    else:
        sampled_indices = label_indices
    balanced_features_list.append(balanced_features[sampled_indices])
    balanced_labels_list.append(balanced_labels[sampled_indices])

# convert the balanced features and labels to numpy arrays
balanced_features = np.vstack(balanced_features_list)
balanced_labels = np.hstack(balanced_labels_list)

# check the label distribution after balancing
balanced_label_distribution = Counter(balanced_labels)
print("Balanced label distribution:", balanced_label_distribution)

X_train, X_test, y_train, y_test = train_test_split(balanced_features, balanced_labels, test_size=0.2, random_state=42, stratify=balanced_labels)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

clf_wrist = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
clf_wrist.fit(X_train, y_train)

y_pred = clf_wrist.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print("Classification Report:")
print(report)

# check the label distribution in training set
label_distribution_train = Counter(y_train)
print("Label distribution in the training set:", label_distribution_train)

# check the label distribution in test set
label_distribution_test = Counter(y_test)
print("Label distribution in the test set:", label_distribution_test)


BVP shape: (4165000, 1)
EDA shape: (4165000, 1)
TEMP shape: (4165000, 1)
BVP shape: (4400200, 1)
EDA shape: (4400200, 1)
TEMP shape: (4400200, 1)
BVP shape: (4393200, 1)
EDA shape: (4393200, 1)
TEMP shape: (4393200, 1)
BVP shape: (4250400, 1)
EDA shape: (4250400, 1)
TEMP shape: (4250400, 1)
BVP shape: (4825799, 1)
EDA shape: (4825799, 1)
TEMP shape: (4825799, 1)
BVP shape: (3563700, 1)
EDA shape: (3563700, 1)
TEMP shape: (3563700, 1)
BVP shape: (3719799, 1)
EDA shape: (3719799, 1)
TEMP shape: (3719799, 1)
BVP shape: (3528700, 1)
EDA shape: (3528700, 1)
TEMP shape: (3528700, 1)
BVP shape: (3740100, 1)
EDA shape: (3740100, 1)
TEMP shape: (3740100, 1)
BVP shape: (3556701, 1)
EDA shape: (3556701, 1)
TEMP shape: (3556701, 1)
BVP shape: (3794000, 1)
EDA shape: (3794000, 1)
TEMP shape: (3794000, 1)
BVP shape: (3763200, 1)
EDA shape: (3763200, 1)
TEMP shape: (3763200, 1)
BVP shape: (3576300, 1)
EDA shape: (3576300, 1)
TEMP shape: (3576300, 1)
BVP shape: (3826200, 1)
EDA shape: (3826200, 1)
TEM

MemoryError: Unable to allocate 119. MiB for an array with shape (15610004,) and data type int64

In [99]:
def extract_and_process_data_chest(subject_id):
    subject_data = wesad_data[subject_id]
    subject_details = subject_data[subject_id]
    chest_data = subject_details['signal']['chest']
    labels = subject_details['label']
    
    # flatten and reshape each signal
    ecg = chest_data['ECG'].flatten().reshape(-1, 1)
    emg = chest_data['EMG'].flatten().reshape(-1, 1)
    eda = chest_data['EDA'].flatten().reshape(-1, 1)
    temp = chest_data['Temp'].flatten().reshape(-1, 1)
    resp = chest_data['Resp'].flatten().reshape(-1, 1)
    
    # print the shapes of all features
    print(f"ECG shape: {ecg.shape}")
    print(f"EMG shape: {emg.shape}")
    print(f"EDA shape: {eda.shape}")
    print(f"TEMP shape: {temp.shape}")
    print(f"RESP shape: {resp.shape}")
    
    # combine the features into a single array
    features = np.hstack((
        ecg,
        emg,
        eda,
        temp,
        resp
    ))

    return features, labels

# extract and process data from subjects S10 and S11
features_chest_s2, labels_chest_s2 = extract_and_process_data_chest('S2')
# features_chest_s15, labels_chest_s15 = extract_and_process_data_chest('S15')
# features_chest_s16, labels_chest_s16 = extract_and_process_data_chest('S16')
# features_chest_s17, labels_chest_s17 = extract_and_process_data_chest('S17')

# combine features and labels from both subjects
combined_features = np.vstack((features_chest_s2))
combined_labels = np.hstack((labels_chest_s2))

l = pd.DataFrame(np.hstack((combined_features, combined_labels.reshape(-1, 1))))
l = l[l[5].isin([1.0, 2.0, 3.0])]
chest_combined_features = l.loc[:,:4]
chest_combined_labels = l.loc[:, 5]

# check initial label distribution
initial_label_distribution = Counter(combined_labels)
print("Initial label distribution:", initial_label_distribution)

X_train_val_chest, X_test_chest, y_train_val_chest, y_test_chest = train_test_split(
    chest_combined_features, chest_combined_labels, test_size=0.2, random_state=42
)

# Then, split the remaining 80% into 80% training and 20% validation
X_train_chest, X_val_chest, y_train_chest, y_val_chest = train_test_split(
    X_train_val_chest, y_train_val_chest, test_size=0.2, random_state=42
)

scaler = StandardScaler()

X_train_chest = scaler.fit_transform(X_train_chest)
X_val_chest = scaler.transform(X_val_chest)
X_test_chest = scaler.transform(X_test_chest)

clf_chest = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')

clf_chest.fit(X_train_chest, y_train_chest)

# Step 3: Validate the model on validation data
y_val_pred_chest = clf_chest.predict(X_val_chest)
val_accuracy = accuracy_score(y_val_chest, y_val_pred_chest)
print(f"Validation Accuracy: {val_accuracy}")

y_test_pred_chest = clf_chest.predict(X_test_chest)

# Step 5: Evaluate the final model on the test set
test_accuracy = accuracy_score(y_test_chest, y_test_pred_chest)
print(f"Test Accuracy: {test_accuracy}")

# Step 6: Generate a detailed classification report for the test set
test_report = classification_report(y_test_chest, y_test_pred_chest)
print("Test Classification Report:")
print(test_report)

ECG shape: (4255300, 1)
EMG shape: (4255300, 1)
EDA shape: (4255300, 1)
TEMP shape: (4255300, 1)
RESP shape: (4255300, 1)
Initial label distribution: Counter({0: 2142701, 1: 800800, 4: 537599, 2: 430500, 3: 253400, 6: 45500, 7: 44800})
Validation Accuracy: 0.999806358186839
Test Accuracy: 0.9998080420286927
Test Classification Report:
              precision    recall  f1-score   support

         1.0       1.00      1.00      1.00    159827
         2.0       1.00      1.00      1.00     86063
         3.0       1.00      1.00      1.00     51050

    accuracy                           1.00    296940
   macro avg       1.00      1.00      1.00    296940
weighted avg       1.00      1.00      1.00    296940



In [89]:
features_chest_s15, labels_chest_s15 = extract_and_process_data_chest('S15')

ECG shape: (3676400, 1)
EMG shape: (3676400, 1)
EDA shape: (3676400, 1)
TEMP shape: (3676400, 1)
RESP shape: (3676400, 1)


In [94]:
# combine features and labels from both subjects
combined_features = np.vstack((features_chest_s15))
combined_labels = np.hstack((labels_chest_s15))

l = pd.DataFrame(np.hstack((combined_features, combined_labels.reshape(-1, 1))))
l = l[l[5].isin([1.0, 2.0, 3.0])]
S15_chest_combined_features = l.loc[:,:4]
S15_chest_combined_labels = l.loc[:, 5]

In [95]:
S15 = scaler.fit_transform(S15_chest_combined_features)

In [96]:
S15_y_pred_chest = clf_chest.predict(S15)

accuracy_chest = accuracy_score(S15_chest_combined_labels, S15_y_pred_chest)
report_chest = classification_report(S15_chest_combined_labels, S15_y_pred_chest)

print(f"Accuracy: {accuracy_chest}")
print("Classification Report:")
print(report_chest)

c:\Users\rusha\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\rusha\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Accuracy: 0.5712635148103128
Classification Report:
              precision    recall  f1-score   support

         1.0       0.55      1.00      0.71    822500
         2.0       1.00      0.15      0.26    480200
         3.0       0.00      0.00      0.00    260400

    accuracy                           0.57   1563100
   macro avg       0.52      0.38      0.32   1563100
weighted avg       0.60      0.57      0.45   1563100



c:\Users\rusha\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Fusion

In [9]:
prob_wrist_train = clf_wrist.predict_proba(X_train)
prob_chest_train = clf.predict_proba(X_train_chest)

ValueError: operands could not be broadcast together with shapes (253400,5) (851060,7) 

In [ ]:
fused_features_train = np.hstack((prob_wrist_train, prob_chest_train))
fused_model = RandomForestClassifier(random_state=42)
fused_model.fit(fused_features_train, y_train)  # i need to ensure y_train matches the labels used in training both models
# Get probabilities on the test set
prob_wrist_test = clf_wrist.predict_proba(X_test)
prob_chest_test = clf.predict_proba(X_test_chest)

# Combine probabilities for test data
fused_features_test = np.hstack((prob_wrist_test, prob_chest_test))

# Predict using the fused model
y_pred_fused = fused_model.predict(fused_features_test)

# Evaluate
accuracy = accuracy_score(y_test, y_pred_fused)
report = classification_report(y_test, y_pred_fused)

print(f"Accuracy: {accuracy}")
print("Classification Report:")
print(report)


In [ ]:
from scipy.stats import mode

# Combine predictions using majority voting
combined_preds = np.vstack((pred_wrist, pred_chest)).T
final_pred_vote, _ = mode(combined_preds, axis=1)
final_pred_vote = final_pred_vote.flatten()

In [ ]:
# Combine predictions as features for stacking
stacked_features = np.column_stack((pred_wrist, pred_chest))

# Train a meta-classifier (e.g., Logistic Regression) on these combined features
meta_clf = LogisticRegression(random_state=42)
meta_clf.fit(stacked_features, y_test)

# Predict using the meta-classifier
final_pred_stack = meta_clf.predict(stacked_features)

In [ ]:
# Average probabilities
avg_prob = (prob_wrist + prob_chest) / 2

# Final prediction by averaging probabilities
final_pred_avg = np.argmax(avg_prob, axis=1)

In [ ]:
import pandas as pd
import numpy as np
from autogluon.tabular import TabularPredictor
from sklearn.model_selection import train_test_split

# Load your data (assuming it's already combined as df_combined)
# df_combined = pd.concat([df_subject_1, df_subject_2, df_subject_3], ignore_index=True)


features = ['acc_x', 'acc_y', 'acc_z', 'bvp', 'eda', 'temperature']
X = df_combined[features]


y = df_combined['label']

# Combine features and target into one DataFrame
df_ag = df_combined[features + ['label']]

train_data, test_data = train_test_split(df_ag, test_size=0.2, random_state=42)

train_data.to_csv('train_data.csv', index=False)
test_data.to_csv('test_data.csv', index=False)

# Load the data using AutoGluon's TabularPredictor
predictor = TabularPredictor(label='label', eval_metric='accuracy').fit(train_data='train_data.csv')
